# Рекомендательная система с помощью нейронной сети

Мы будем использовать модуль tensorflow, в котором реализовано много полезных методов для имплементации (внедрения) нейронных сетей. Установим его:

```python
    pip install tensorflow
```

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
# Импортируем необходимые слои и класс модели из Keras (модуля TensorFlow)
from tensorflow.keras.layers import Input         # Класс для задания входного слоя модели (точка входа данных)
from tensorflow.keras.layers import Embedding     # Слой для преобразования целых числовых индексов в плотные векторы фиксированной размерности (эмбеддинги)
from tensorflow.keras.layers import Flatten       # Слой для преобразования многомерного тензора в одномерный (например, чтобы передать в полносвязный слой)
from tensorflow.keras.layers import Dot           # Слой для вычисления скалярного произведения (dot product) двух векторов (часто используется для рекомендаций)
from tensorflow.keras.layers import Dense         # Полносвязный (dense) слой нейронной сети
from tensorflow.keras.layers import Concatenate   # Слой для объединения нескольких входов по последней размерности (склейка векторов)

from tensorflow.keras.models import Model         # Класс для создания и управления полной моделью (объединяет входы, слои и выходы)

2025-07-07 11:07:23.531604: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Мы будем использовать данные из предыдущего примера, но лишь те, которые содержат информацию об оценках, выставленных книгам пользователями. 

Загрузим данные:

In [2]:
ratings_df = pd.read_csv('Data/good_dread_books/ratings.csv')
ratings_df.head(5)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


Разобьем данные на обучающую и тестовую выборки в отношении 4:1. В качестве значения параметра random_state возьмите число 42.

Проверим сколько объектов находится в обучающей выборке.

In [3]:
train, test = train_test_split(
    ratings_df, # Общая выборка
    test_size=0.2, # размер тестовой выборки 20%
    random_state=42
)

In [4]:
display(train.shape)

(785404, 3)

Предварительно вычислим количество уникальных пользователей и книг:

In [6]:
n_books = ratings_df['book_id'].nunique()
print(n_books)

n_users = ratings_df['user_id'].nunique()
print(n_users)

10000
53424


В первую очередь необходимо создать эмбеддинги для книг и пользователей. Создаём эмбеддинги для книг:

In [7]:
book_input = Input(shape=[1], name='Book-Input')
book_embedding = Embedding(n_books+1, 5, name='Book-Embedding')(book_input)
book_vec = Flatten(name='Flatten-Books')(book_embedding)

Сначала мы задаём размерность входного слоя (в этом параметре макс. значение всегда равно длинне вектора + 1). После этого определяем размер эмбеддинга — в данном случае снижаем размерность до 5. Далее мы разворачиваем результат в массив с одним измерением с помощью слоя Flatten().

Делаем то же самое для пользователей:

In [8]:
user_input = Input(shape=[1], name="User-Input")
user_embedding = Embedding(n_users+1, 5, name="User-Embedding")(user_input)
user_vec = Flatten(name="Flatten-Users")(user_embedding)

Теперь, когда мы создали представления как для книг, так и для пользователей, нам необходимо их соединить:

In [9]:
conc = Concatenate()([book_vec, user_vec])

Далее начинаем «собирать» нашу нейронную сеть из слоёв! Dense обозначает полносвязный слой. Также мы обозначаем для него количество нейронов и данные, которые идут на вход.

In [10]:
# Первый полносвязный (Dense) слой с 128 нейронами и функцией активации ReLU.
# Этот слой получает на вход объединённые признаки (conc), извлекая из них скрытые зависимости.
fc1 = Dense(128, activation='relu')(conc)

# Второй полносвязный слой с 32 нейронами и функцией активации ReLU.
# Служит для дальнейшего уменьшения размерности признаков и выявления сложных паттернов.
fc2 = Dense(32, activation='relu')(fc1)

# Выходной слой с одним нейроном.
# Так как активация не указана, используется линейная активация по умолчанию.
# Подходит для регрессии (например, предсказания рейтинга).
out = Dense(1)(fc2)

Собираем модель — передаём входные данные для книг и пользователей, а также архитектуру нейронной сети:

In [11]:
model2 = Model([user_input, book_input], out)

Также нам необходимо задать алгоритм оптимизации и метрику, которую мы будем оптимизировать. В данном случае будем использовать метод adam (одна из вариаций градиентного спуска) и хорошо известную среднеквадратичную ошибку MSE:

In [12]:
model2.compile(optimizer='adam', loss='mean_squared_error')

# Теперь будем обучать нашу модель:
history = model2.fit([train.user_id, train.book_id], train.rating, epochs=5, verbose=1)

Epoch 1/5
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 61s 2ms/step - loss: 1.0113
Epoch 2/5
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 47s 2ms/step - loss: 0.6809
Epoch 3/5
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 45s 2ms/step - loss: 0.6510
Epoch 4/5
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 44s 2ms/step - loss: 0.6225
Epoch 5/5
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 52s 2ms/step - loss: 0.6008


В параметр эпох передаём значение 5: у нас будет реализовано пять эпох — пять обучений нейронной сети. На каждой из эпох обновляются веса для минимизации ошибки.

Теперь можно оценить качество:

In [14]:
model2.evaluate([test.user_id, test.book_id], test.rating)

6136/6136 ━━━━━━━━━━━━━━━━━━━━ 6s 898us/step - loss: 0.7806


0.7806089520454407

Обычно для улучшения качества модели каким-то образом модифицируют нейронную сеть: дополняют её, увеличивают время обучения. Добавим ещё один полносвязный слой с восемью нейронами после полносвязного слоя с 32 нейронами. Обучим нейронную сеть, реализовав десять эпох:

In [13]:
fc1 = Dense(128, activation='relu')(conc)
fc2 = Dense(32, activation='relu')(fc1)
# Третий полносвязный слой с 8 нейронами и функцией активации ReLU.
# Продолжает сжимать пространство признаков и готовит данные для выхода.
fc3 = Dense(8, activation='relu')(fc2)
out = Dense(1)(fc3)

model2 = Model([user_input, book_input], out)
# Компиляция модели: используем оптимизатор Adam и функцию потерь — среднеквадратичную ошибку (MSE),
# так как это задача регрессии.
model2.compile('adam', 'mean_squared_error')
# Обучение модели: подаём списки ID пользователей и книг для обучения,
# а также соответствующие рейтинги. Обучаем 10 эпох.
result = model2.fit([train.user_id, train.book_id], train.rating, epochs=10, verbose=1)
# Оценка модели на тестовой выборке: вычисляем значение функции потерь (MSE)
# на тестовых данных.
model2.evaluate([test.user_id, test.book_id], test.rating)

Epoch 1/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 63s 2ms/step - loss: 0.7546
Epoch 2/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 58s 2ms/step - loss: 0.5649
Epoch 3/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 56s 2ms/step - loss: 0.5421
Epoch 4/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 53s 2ms/step - loss: 0.5239
Epoch 5/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - loss: 0.5088
Epoch 6/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 53s 2ms/step - loss: 0.4955
Epoch 7/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - loss: 0.4865
Epoch 8/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 55s 2ms/step - loss: 0.4780
Epoch 9/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 57s 2ms/step - loss: 0.4704
Epoch 10/10
24544/24544 ━━━━━━━━━━━━━━━━━━━━ 52s 2ms/step - loss: 0.4648
6136/6136 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.7806


0.7806089520454407

# Вывод:

Качество получившейся модели не будет выше качества предыдущей, так как усложнение сети или увеличение количества эпох не всегда даёт высокое качество.